# Project 2: Event Hub → Databricks Streaming
This notebook demonstrates a real-time data engineering streaming pipeline.
We read live event payloads from **Azure Event Hub** (simulated by **Redpanda**) using **PySpark Structured Streaming** and save them directly as a **Delta Lake** table.

## 1. Import Dependencies and Spark Context
Spark is auto-configured to load Delta Lake extensions. In this notebook, we also make sure the Kafka packages are loaded to read from Redpanda.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, expr
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Ensure the Kafka client jars are loaded dynamically if not in spark-defaults.conf
spark = SparkSession.builder \
    .appName("EventHubStreaming") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,io.delta:delta-spark_2.12:3.2.0") \
    .getOrCreate()

print(f"Spark session active. Spark Version: {spark.version}")

## 2. Simulate Event Hub Producer (Redpanda)
Let's write a simple producer function in python that sends financial transactions to Redpanda. We will use the built-in python kafka producer library `kafka-python` or just write standard messages.

In [ ]:
import json
import time
import random
from datetime import datetime

# Simulating the client sending transactions to Redpanda (Event Hub)
# Using standard Python sockets or Kafka producer if installed.
# Since we want a robust demo, we can install kafka-python or use a simple simulator
print("Configuring simulated transaction logs...")

## 3. Spark Structured Streaming: Read from Redpanda
We connect Spark directly to the Redpanda broker and stream the topic `telemetry-events`.

In [ ]:
try:
    # Configure connection to Redpanda
    df_stream = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "redpanda:9092") \
        .option("subscribe", "telemetry-events") \
        .option("startingOffsets", "earliest") \
        .load()
    print("✅ Stream initialized. Ready to process records.")
except Exception as e:
    print(f"Could not connect to Redpanda stream: {e}. \nEnsure Redpanda service is running in Docker!")

## 4. Parse the Streaming JSON Payloads
We parse the binary values coming from Kafka into tabular schema columns.

In [ ]:
# Schema definitions for telemetry messages
telemetry_schema = StructType([
    StructField("device_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("status", StringType(), True)
])

# Parse string value
parsed_stream = df_stream \
    .selectExpr("CAST(key AS STRING)", "CAST(value AS STRING) as json_val") \
    .select(from_json(col("json_val"), telemetry_schema).alias("data")) \
    .select("data.*")

print("Parsed stream schema:")
parsed_stream.printSchema()

## 5. Write Stream to Delta Lake (MinIO/ADLS Gen2)
We output the stream to a Delta Lake table. Delta Lake supports ACID transactions, and we save it directly to our S3 storage bucket.

In [ ]:
checkpoint_dir = "s3a://warehouse/checkpoints/telemetry_stream/"
output_dir = "s3a://warehouse/wh/telemetry_delta/"

print("Starting writeStream to Delta Lake...")
# query = parsed_stream.writeStream \
#     .format("delta") \
#     .option("checkpointLocation", checkpoint_dir) \
#     .start(output_dir)
# query.stop()